# Coherence Under Change: real-model runs on Colab (Phase 4 pilot + Phase 5 fine-tuning)

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Upload the whole `coherence-under-change` folder to
the top of your Google Drive (`MyDrive/coherence-under-change`).

Everything is **resumable**: each finished (model, bed, condition, sequence) run is saved to Drive as soon as it
ends. If Colab disconnects, reconnect, re-run cells 1-3 (cell 2 sets the model) and re-run the same run cell; finished runs are skipped.

The judge in these runs is the project's reference engine (validated against Batfish: see `scripts/validate_*`).

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Google Drive + project folder (code is copied locally for speed; results go to Drive)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
SRC = '/content/drive/MyDrive/coherence-under-change'
assert os.path.isdir(SRC), 'Upload the coherence-under-change folder to MyDrive first'
shutil.rmtree('/content/cuc', ignore_errors=True)
shutil.copytree(SRC, '/content/cuc', ignore=shutil.ignore_patterns('runs', 'results', '__pycache__', '*.png'))
OUT = '/content/drive/MyDrive/cuc_runs'          # all model runs land here
os.makedirs(OUT, exist_ok=True)
MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'   # change both lines to run another model
NAME = 'qwen2.5-coder-7b'
%cd /content/cuc

In [ ]:
# 3. Libraries (about 2 minutes)
!pip -q install -U transformers accelerate bitsandbytes peft trl datasets networkx

In [ ]:
# 4. Optional: Hugging Face login - only needed for gated models such as Llama 3.1 (request access on its model page first)
# from huggingface_hub import login; login()

In [ ]:
# 5. Pipeline check without a model (seconds): gold patches must all succeed
!python3 -m unittest discover -s tests 2>&1 | tail -3
!python3 scripts/run_models.py --backend gold --bed abilene --seeds 100-100 --length 10 --conds C0,C3 --out /content/check

## Phase 4 pilot

Model: **Qwen2.5-Coder-7B-Instruct** in 4-bit (no access request needed). The first cell is a speed test on 4 steps;
use its timing to decide the pilot size. The pilot itself is 3 sequences x 15 steps on the Abilene bed, conditions
C0 (no guard) and C3 (full guard) first, then C1 and C2.  `--workers 3` runs the 3 sequences in parallel and batches
their requests on the GPU.

In [ ]:
# 6. Speed test: 1 sequence, 4 steps, C0 (loads the model: about 3-5 minutes the first time)
# MODEL and NAME are set in cell 2
!python3 scripts/run_models.py --backend hf --model {MODEL} --name {NAME}-speedtest --bed abilene --seeds 100-100 --length 4 --conds C0 --out /content/speedtest

In [ ]:
# 7. Pilot part 1: C0 and C3 (resumable - just re-run this cell after a disconnect)
!python3 scripts/run_models.py --backend hf --model {MODEL} --name {NAME} --bed abilene --seeds 100-102 --length 15 --conds C0,C3 --workers 3 --out {OUT}

In [ ]:
# 8. Pilot part 2: C1 and C2
!python3 scripts/run_models.py --backend hf --model {MODEL} --name {NAME} --bed abilene --seeds 100-102 --length 15 --conds C1,C2 --workers 3 --out {OUT}

In [ ]:
# 9. Results table + paired bootstrap (also saved to Drive: cuc_runs/summary/)
!python3 scripts/summarize_runs.py {OUT} --out {OUT}/summary

## Optional: a second baseline model

Same cells with another model, e.g. `meta-llama/Llama-3.1-8B-Instruct` (needs the access request + cell 4), or
`mistralai/Mistral-7B-Instruct-v0.3`. Change `MODEL` and `NAME` in cell 2, then re-run cells 2 and 6-9.

## Phase 5: QLoRA fine-tuning

Training data comes only from the **training beds** (Abilene, BtEurope) and **training seeds** (1000+). The held-out
test bed is **Cesnet**. The data script runs a leakage check and stops if anything from the test side appears in the
training data. Training on a T4 is slow; `N_SEQ_TRAIN = 12` gives roughly 1,200 examples (about 1-2 hours).

In [ ]:
# 10. Build training data + leakage check
N_SEQ_TRAIN = 12
!python3 scripts/make_sft_data.py --n-seq {N_SEQ_TRAIN} --length 50 --out /content/sft

In [ ]:
# 11. QLoRA training (adapter saved to Drive)
import json, torch, os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

ADAPTER = f'/content/drive/MyDrive/cuc_adapters/{NAME}-qlora'
MAX_LEN = 3072
tok = AutoTokenizer.from_pretrained(MODEL)
def to_pc(ex):                     # prompt-completion format: loss only on the answer
    return {'prompt': [ex['messages'][0]], 'completion': [ex['messages'][1]]}
ds = load_dataset('json', data_files={'train': '/content/sft/train.jsonl', 'val': '/content/sft/val.jsonl'})
ds = ds.map(to_pc, remove_columns=ds['train'].column_names)
fits = lambda ex: len(tok.apply_chat_template(ex['prompt'] + ex['completion'], tokenize=True)) <= MAX_LEN
ds = ds.filter(fits)
print({k: len(v) for k, v in ds.items()})

model = AutoModelForCausalLM.from_pretrained(MODEL, device_map='auto', quantization_config=BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16))
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules='all-linear', task_type='CAUSAL_LM')
cfg = dict(output_dir='/content/qlora_ckpt', num_train_epochs=1, per_device_train_batch_size=1,
           gradient_accumulation_steps=8, learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03,
           logging_steps=10, eval_strategy='steps', eval_steps=100, save_steps=100, fp16=True,
           gradient_checkpointing=True, report_to='none')
try:
    args = SFTConfig(max_length=MAX_LEN, **cfg)
except TypeError:                  # older TRL
    args = SFTConfig(max_seq_length=MAX_LEN, **cfg)
trainer = SFTTrainer(model=model, args=args, train_dataset=ds['train'], eval_dataset=ds['val'],
                     peft_config=lora, processing_class=tok)
trainer.train()
trainer.model.save_pretrained(ADAPTER); tok.save_pretrained(ADAPTER)
print('adapter saved to', ADAPTER)

In [ ]:
# 12. Free the GPU, then evaluate base vs fine-tuned on the HELD-OUT bed (cesnet), same sequences
import gc, torch
for v in ('trainer', 'model'):
    globals().pop(v, None)
gc.collect(); torch.cuda.empty_cache()
!python3 scripts/run_models.py --backend hf --model {MODEL} --name {NAME} --bed cesnet --seeds 100-102 --length 15 --conds C0,C3 --workers 3 --out {OUT}
!python3 scripts/run_models.py --backend hf --model {MODEL} --adapter {ADAPTER} --name {NAME}-ft --bed cesnet --seeds 100-102 --length 15 --conds C0,C3 --workers 3 --out {OUT}
!python3 scripts/summarize_runs.py {OUT} --out {OUT}/summary